# Task 82: Manual Multiple Imputation Comparison

Compare manually coded KNN, iterative chained conditional prediction, and one-pass predictive imputation using a seeded artificial mask because the source data has no missing values.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

paths=[Path('heart.csv'),Path('Task_82_Manual_Multiple_Imputation/heart.csv'),Path('Heart_Disease_Tasks/Task_82_Manual_Multiple_Imputation/heart.csv')]
data=pd.read_csv(next(path for path in paths if path.is_file())).drop_duplicates().reset_index(drop=True)
features=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
values=data[features].to_numpy(float)
categorical={'sex','cp','fbs','restecg','exang','slope','ca','thal'}
actual_missing=int(np.isnan(values).sum())
print('Missing values in source feature matrix:',actual_missing)
if actual_missing:
    original=values.copy();observed=np.isfinite(values)
else:
    rng=np.random.default_rng(82);original=values.copy();observed=rng.random(values.shape)>=.12
masked=original.copy();masked[~observed]=np.nan

def initialize(matrix,mask):
    filled=matrix.copy();category_values={}
    for j,name in enumerate(features):
        present=matrix[mask[:,j],j]
        if name in categorical:
            classes,counts=np.unique(present,return_counts=True);category_values[j]=classes
            filled[~mask[:,j],j]=classes[np.argmax(counts)]
        else:filled[~mask[:,j],j]=np.median(present)
    return filled,category_values

def conditional_predict(filled,mask,j,name,category_values,ridge=1.):
    known=mask[:,j];missing=~known
    if not missing.any() or not known.any():return filled[:,j]
    columns=[k for k in range(filled.shape[1]) if k!=j]
    xtrain=filled[known][:,columns];xquery=filled[missing][:,columns];mu=xtrain.mean(0);sd=xtrain.std(0);sd[sd==0]=1
    xtrain=(xtrain-mu)/sd;xquery=(xquery-mu)/sd;target=filled[known,j]
    if name in categorical:
        classes=category_values[j]
        centers=np.array([xtrain[target==value].mean(0) if np.any(target==value) else xtrain.mean(0) for value in classes])
        pred=classes[np.argmin(np.sum((xquery[:,None,:]-centers[None,:,:])**2,axis=2),axis=1)]
    else:
        design=np.column_stack([np.ones(len(xtrain)),xtrain]);query=np.column_stack([np.ones(len(xquery)),xquery]);penalty=np.eye(design.shape[1])*ridge;penalty[0,0]=0
        beta=np.linalg.solve(design.T@design+penalty,design.T@target);pred=query@beta
    result=filled[:,j].copy();result[missing]=pred;return result

def impute_predictive(matrix,mask):
    filled,categories_by_column=initialize(matrix,mask)
    for j,name in enumerate(features):filled[:,j]=conditional_predict(filled,mask,j,name,categories_by_column)
    return filled

def impute_mice(matrix,mask,iterations=4):
    filled,categories_by_column=initialize(matrix,mask)
    for _ in range(iterations):
        for j,name in enumerate(features):filled[:,j]=conditional_predict(filled,mask,j,name,categories_by_column)
    return filled

def impute_knn(matrix,mask,k=7):
    filled,_=initialize(matrix,mask);sd=np.nanstd(np.where(mask,matrix,np.nan),axis=0);sd[~np.isfinite(sd)|(sd==0)]=1
    for row,j in zip(*np.where(~mask)):
        candidates=np.flatnonzero(mask[:,j]);candidates=candidates[candidates!=row]
        common=mask[candidates]&mask[row][None,:];common[:,j]=False;delta=(filled[candidates]-filled[row])/sd
        distance=np.sqrt(np.sum(np.where(common,delta**2,0),axis=1)/np.maximum(common.sum(1),1));neighbors=candidates[np.argsort(distance,kind='stable')[:min(k,len(candidates))]]
        neighbor_values=matrix[neighbors,j]
        if features[j] in categorical:
            classes,counts=np.unique(neighbor_values,return_counts=True);filled[row,j]=classes[np.argmax(counts)]
        elif len(neighbors):filled[row,j]=float(neighbor_values.mean())
    return filled

imputations={'KNN k=7':impute_knn(masked,observed),'MICE-like chained equations':impute_mice(masked,observed,iterations=4),'One-pass predictive':impute_predictive(masked,observed)}
records=[]
for j,name in enumerate(features):
    holdout=~observed[:,j]
    if not holdout.any():continue
    for method,completed in imputations.items():
        if name in categorical:error=float(np.mean(completed[holdout,j]!=original[holdout,j]));metric='Misclassification rate'
        else:
            iqr=float(np.quantile(original[:,j],.75)-np.quantile(original[:,j],.25)) or 1.
            error=float(np.mean(np.abs(completed[holdout,j]-original[holdout,j]))/iqr);metric='IQR-normalized MAE'
        records.append({'Feature':name,'Method':method,'Metric':metric,'Error':error})
results=pd.DataFrame(records)
selected=results.loc[results.groupby('Feature')['Error'].idxmin()].sort_values('Feature')
print('Per-method artificial-mask validation errors:');display(results.pivot(index='Feature',columns='Method',values='Error').round(3))
print('Lowest validation-error method per column:');display(selected.set_index('Feature').round(3))
print('Masked cells evaluated:',int((~observed).sum()),'of',observed.size)
print('Actual source missingness:',actual_missing,'; source file remains unchanged.')
print('MICE-like categorical updates use class-centroid prediction; continuous updates use chained ridge regressions. Re-run multiple masks before selecting a production imputer.')

Missing values in source feature matrix: 0
Per-method artificial-mask validation errors:


Method,KNN k=7,MICE-like chained equations,One-pass predictive
Feature,,,
age,0.462,0.460,0.440
ca,0.366,0.537,0.561
chol,0.742,0.562,0.581
cp,0.512,0.537,0.561
exang,0.188,0.188,0.219
fbs,0.108,0.270,0.243
oldpeak,0.470,0.461,0.456
restecg,0.500,0.523,0.614
sex,0.462,0.538,0.538


Lowest validation-error method per column:


,Method,Metric,Error
Feature,,,
age,One-pass predictive,IQR-normalized MAE,0.440
ca,KNN k=7,Misclassification rate,0.366
chol,MICE-like chained equations,IQR-normalized MAE,0.562
cp,KNN k=7,Misclassification rate,0.512
exang,KNN k=7,Misclassification rate,0.188
fbs,KNN k=7,Misclassification rate,0.108
oldpeak,One-pass predictive,IQR-normalized MAE,0.456
restecg,KNN k=7,Misclassification rate,0.500
sex,KNN k=7,Misclassification rate,0.462


Masked cells evaluated: 479 of 3926
Actual source missingness: 0 ; source file remains unchanged.
MICE-like categorical updates use class-centroid prediction; continuous updates use chained ridge regressions. Re-run multiple masks before selecting a production imputer.
